# 第22讲：Llama generate完整流程

原单元从0编号；原代码、Markdown与历史输出保留。完整generate与CPU教学验证是明确补充，未加载原7B权重。


## 原单元 0（code）

【必须会】text是Python str，内容为I believe the meaning of life is。它还不是Tensor，也没有batch轴；模型稍后接收的是整数token ID。赋值本身不生成文本。


In [1]:
# 输入文本是str，还不是token Tensor。
text = "I believe the meaning of life is"

## 原单元 1（code）

【必须会】导入SentencePieceProcessor并从./tokenizer.model建立实例。相对路径取决于当前工作目录，应从代码/tutorials启动内核。这里加载的是分词模型，不是Llama神经网络权重。


In [2]:
# 读取真实SentencePiece模型，相对路径取决于工作目录。
from sentencepiece import SentencePieceProcessor
sp_model = SentencePieceProcessor(model_file='./tokenizer.model')

## 原单元 2（code）

【必须会】四次print读取真实特殊ID/词表：BOS=1、EOS=2、PAD=-1、词表32000。PAD=-1是占位哨兵，不是合法embedding索引；完整循环必须在负ID被送入embedding之前填好有效前缀。这里的ID不能直接套到其他tokenizer。


In [13]:
# 打印BOS/EOS/PAD及词表大小，PAD=-1只作占位。
print(sp_model.bos_id())
print(sp_model.eos_id())
print(sp_model.pad_id())
print(sp_model.vocab_size())

1
2
-1
32000


## 原单元 3（code）

【必须会】encode(text)返回7个普通整数；列表中的*把它们展开，前置BOS后得到[1,306,4658,278,6593,310,2834,338]。末表达式在notebook显示列表。它是List[int]单条提示；generate要求List[List[int]]批次，text_completion会负责把每条文本编码后组成batch，直接把本变量传给generate会报类型错误。


In [6]:
# *展开编码ID；这是单条列表，generate需要再包成二维batch。
prompt_tokens = [sp_model.bos_id(), *sp_model.encode(text)]
prompt_tokens

[1, 306, 4658, 278, 6593, 310, 2834, 338]

## 原单元 4（markdown）

【必须会】这是self.generate的调用片段，generation_tokens/generation_logprobs用二元解包接收返回值。默认echo=False，仅返回生成部分；True从开头保留prompt，不过末尾仍会按EOS截断。返回token IDs，不是已decode的str。logprobs=False时第二项是None。

max_gen_len是每条提示词最多保留的新增数量，total_len=min(max_seq_len,max_gen_len+max_prompt_len)是整个批次统一容量。max_prompt_len取最长提示，而循环从最短提示长度开始。8+64=72只是示例，实际还受容量限制；参数max_seq_len不是Llama2全部型号永远固定1024或2048的定义。

```python
# 调用generate并解包两个返回项。
generation_tokens, generation_logprobs = self.generate(
    prompt_tokens=prompt_tokens,
    max_gen_len=max_gen_len,
    temperature=temperature,
    top_p=top_p,
    logprobs=logprobs,
    echo=echo,
)
```



```
generation_tokens, generation_logprobs = self.generate(
    prompt_tokens=prompt_tokens,
    max_gen_len=max_gen_len,
    temperature=temperature,
    top_p=top_p,
    logprobs=logprobs,
    echo=echo,
)
```

- `echo (bool, optional): `
    - `Flag indicating whether to include prompt tokens in the generated output. Defaults to False.`
- `total_len = min(params.max_seq_len, max_gen_len + max_prompt_len)`
    - `params.max_seq_len`: 1024 (2048)
    - max_gen_len + max_prompt_len = 64 + 8 = 72

## 原单元 5（markdown）

【建议会】basics把前几讲的模型模块串联起来。本讲重点为生成控制流，不额外发明训练loss、optimizer或数据集。


## basics

## 原单元 6（markdown）

【建议会】列出RMSNorm/SwiGLU、两段RoPE、KV cache、GQA与采样的相关视频。来源链接保留；nuclear sampling应校正为nucleus sampling，中文核采样。这个generate实现有temperature和top_p，没有实现top_k参数，不能把视频标题当成本函数签名。


- RMSNorm 与 SwiGLU：https://www.bilibili.com/video/BV1e14y1C7G8
- RoPE 相对位置编码：
    - https://www.bilibili.com/video/BV1Dh4y1P7KY/
    - https://www.bilibili.com/video/BV18u411M7j1/
- cache KV：https://www.bilibili.com/video/BV1FB4y1Z79y/
- GQA，Grouped Query Attention：https://www.bilibili.com/video/BV1vc411o7fa/
- top_p/top_k，nuclear sampling 核采样
    - https://www.bilibili.com/video/BV1Ho4y1x76q

## 原单元 7（markdown）

【必须会】Llama.generate是Llama实例的方法，使用self.model和self.tokenizer。完整方法有@torch.inference_mode()，关闭梯度记录，但不会自动调用model.eval()。构造/加载模型和设置推理状态是其他环节；本次不执行需要NCCL/checkpoint的build。


## `Llama.generate`

## 原单元 8（markdown）

【必须会】表中heads是Q heads；D/Hq都得到128。模型层数/通道影响一次forward，生成循环的起止位置由token长度决定，不由head数决定。70B的KV heads不能从heads列直接读成64，分组见第21讲。


| **model** | **heads** | **layers** | **dim** |   **head_dim**   |
|-----------|-----------|------------|---------|------------------|
| 7b        | 32        | 32         | 4096    | 4096/32=128      |
| 13b       | 40        | 40         | 5120    | 5120/40=128      |
| 70b       | 64        | 80         | 8192    | 8192/64=128      |

## 原单元 9（markdown）

【必须会】标题省略了切片细节，真正调用为model.forward(tokens[:,prev_pos:cur_pos],prev_pos)。模型返回浮点[B,S,V]，其中S是本次送入长度，不是累计输出长度；循环用logits[:,-1]的词表向量预测当前位置cur_pos的token。


### `logits = self.model.forward(tokens, prev_pos)`

## 原单元 10（markdown）

【必须会】range(min_prompt_len,total_len)右端不包含。prev_pos初始0，第一次送[0:min_prompt_len]，随后完整方法在写token后令prev_pos=cur_pos，下一次才成为单token切片。单条8-token prompt的调用是[0:8]、[8:9]、[9:10]。

本片段缺prev_pos更新、写token、采样和EOS逻辑，按字面独立执行会一直用prev_pos=0重算完整前缀，不能冒称它自己已实现增量。原句“第一次prompt整体进来”只对单条/等长batch成立，不等长batch首次只处理最短长度，长提示的剩余token由input_text_mask逐位置保护。多token首次prefill需因果上三角mask；后续单token没有chunk内部未来位置。

```python
# 从最短prompt末尾开始，右端total_len不包含。
for cur_pos in range(min_prompt_len, total_len):
    # logits.shape: [bsz, slice_seq_len, vocab_size]
    # tokens: [0, 8), prompt 部分,
    # 下一次 tokens: [8, 9)
    # 下一次 tokens: [9, 10)
    # 仅传尚未处理的[prev_pos:cur_pos]，起始绝对位置为prev_pos。
    logits = self.model.forward(tokens[:, prev_pos:cur_pos], prev_pos)
```



```
for cur_pos in range(min_prompt_len, total_len):
    # logits.shape: [bsz, slice_seq_len, vocab_size]
    # tokens: [0, 8), prompt 部分, 
    # 下一次 tokens: [8, 9)
    # 下一次 tokens: [9, 10)
    logits = self.model.forward(tokens[:, prev_pos:cur_pos], prev_pos)
```

- input prompt tokens 在第一次 forward的时候，整体进来的，需要进行上三角的 mask；

## 原单元 11（markdown）

【建议会】这是手写的累计文本示意，原文meanning拼写与text格不同；保留原文，并不认为这些续写是本次或原7B实际输出。真实过程先产生token ID，token边界不一定等于一个英文单词，最后decode才得到文本。


- I believe the meanning of life is 
- I believe the meanning of life is to 
- I believe the meanning of life is to find
- I believe the meanning of life is to find the
- I believe the meanning of life is to find the happiness
- I believe the meanning of life is to find the happiness that
- I believe the meanning of life is to find the happiness that comes 
- I believe the meanning of life is to find the happiness that comes from

## 原单元 12（markdown）

【必须会】进入缓存环节。KV cache属于每层Attention，不是generate在Python列表中拼接的隐藏状态。tokens保存输入/输出整数，cache保存带位置的K及未旋转V；两者内容、dtype、shape都不同。


### cache KV

## 原单元 13（markdown）

【必须会】完整接口接受[B,S]，S不总等于1：首次prefill通常S>1，随后decode常为1。原“每次只接受一个token”的说法需要限定后续decode。切片读取cache[:B,:P+S]包含此前历史与本次新KV，不包含未用容量尾部。cache本体仍是[Bmax,Tmax,Hkv_local,d]。

两条keys/values语句属于Attention.forward，其中self是Attention；调用generate时self却是Llama。相同self名字并不代表同一个对象。摘录需要提供bsz/start_pos/seqlen等上下文，不能在Llama实例上直接取cache_k。

```python
# 截止到当前（包含历史）！！！
keys = self.cache_k[:bsz, : start_pos + seqlen]
values = self.cache_v[:bsz, : start_pos + seqlen]
```



- token by token 的生成
- 接口层面，`logits = self.model.forward(tokens[:, prev_pos:cur_pos], prev_pos)`
    - 一次只接受一个 token
    - 历史 token(input prompts + 截止到当前生成的 tokens) 信息缓存在 cache_KV 中
    - Attention layer

        ```
        # 截止到当前（包含历史）！！！
        keys = self.cache_k[:bsz, : start_pos + seqlen]
        values = self.cache_v[:bsz, : start_pos + seqlen]
        ```

## 原单元 14（markdown）

【建议会】区分返回控制（logprobs/echo）和选词控制（temperature/top_p）。改变echo不改变同一随机种子下的已生成ID，只改变最后切片；logprobs理论上是概率报告，但固定旧版存在记录顺序缺陷，见本讲实测。


### 其他控制参数

## 原单元 15（markdown）

【必须会】温度公式为p_i(T)=exp(z_i/T)/sum_j exp(z_j/T)，z_i是当前最后位置第i个词表logit，T>0，沿词表归一化。正温度改变分布尖锐度，理论上不改变最大logit索引；熵在非退化例子中随T增加。T=0不能直接代入除法，代码走argmax；负温度也落入else，不做负T softmax。

top_p是在温度softmax之后按概率降序保留动态候选集合、重归一化再采样，不是固定top_k。输出token_logprobs的目标是原模型的log概率，不能直接等同于温度/核截断后采样分布概率；旧版把尚为PAD的新增target先送入CE，实际新增返回-0，不能当真实概率。


- 返回相关：
    - `token_logprobs`
    - `echo`
- 生成相关 
    - `temperature`：温度，温度越高越平均（随机性越高，entropy越高），越低越sharp（确定性越高，entropy 越低）
        $$
        \frac{\exp\left(\frac{z_i}{T}\right)}{\sum_j\exp\left(\frac{z_j}T\right)}
        $$
    - `top_p`：核采样；https://www.bilibili.com/video/BV1Ho4y1x76q

## 原单元 16（markdown）

【必须会】if temperature>0时，logits[:,-1]从[B,S,V]取最后位置得到[B,V]；除标量温度，softmax(dim=-1)沿词表得到概率，sample_top_p返回int64[B,1]。else下torch.argmax(...,dim=-1)返回[B]，所以完整generate随后reshape(-1)统一为[B]。

sample_top_p先sort得到概率与原词表索引，cumsum得到前缀和，用(cumsum-current_prob)>p遮掉候选，原地重归一化，multinomial实际采样，再gather映射回原token ID。使用的是“当前候选之前的累计质量”，因此保留跨过阈值的那一项，恰好等于阈值时也继续保留下一项。它未主动校验p范围；负p可能导致全零/NaN错误。原片段不包含提示词保护、cache更新或停止逻辑，完整流程见官方方法注释。

```python
# 正温度才softmax采样；0或负值走greedy。
if temperature > 0:
    # temperature 与 top_p 组合使用
    # 取最后位置词表分布[B,V]，温度缩放后归一化。
    probs = torch.softmax(logits[:, -1] / temperature, dim=-1)
    # 真实核采样，返回词表ID[B,1]。
    next_token = sample_top_p(probs, top_p)
else:
    # greedy直接选最大logit，[B]；不使用top_p。
    next_token = torch.argmax(logits[:, -1], dim=-1)
```



- temperature & top_p

```
if temperature > 0:
    # temperature 与 top_p 组合使用
    probs = torch.softmax(logits[:, -1] / temperature, dim=-1)
    next_token = sample_top_p(probs, top_p)
else:
    next_token = torch.argmax(logits[:, -1], dim=-1)
```